In [1]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
#Pandas - Pandas is a powerful framework to work with structured data like CSV, Tables, and Data base
#Pandas - Pandas is a powerful framework to work with structured data like CSV, Tables, and Data base
try:
    dataset = pd.read_csv("insurance_pre.csv") # Method to import csv data into 'pd' object
    print("Data loaded successfully")
    print(f"Rows: {len(dataset)}, Columns: {len(dataset.columns)}")
except Exception as e:
    print(f" Either file not found or Unable to open or Unexpected error: {e}")


Data loaded successfully
Rows: 1338, Columns: 6


In [5]:
dataset.columns

Index(['age', 'sex', 'bmi', 'children', 'smoker', 'charges'], dtype='object')

In [7]:
#In Pandas, the get_dummies() function converts categorical variables into dummy/indicator variables (known as one-hot encoding)
#Methods avilable to this 1. One Hot Encoding 2. Using Sckit learn Label Encoder 3. Direct Lable Encoding
dataset=pd.get_dummies(dataset, columns=['sex', 'smoker'], drop_first=True)

In [9]:
dataset.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='object')

In [11]:
independent=dataset[['age', 'bmi', 'children','sex_male', 'smoker_yes']]
dependent=dataset[['charges']]

In [15]:
#split into training set and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(independent, dependent, test_size = 1/3, random_state = 0)

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

pipeline = Pipeline([
    ("dtr", DecisionTreeRegressor(random_state=0))
])

In [39]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "dtr__max_depth": [None, 2, 3, 4, 5, 6, 7, 8],
    "dtr__min_samples_split": [2, 5, 10, 20],
    "dtr__min_samples_leaf": [1, 2],
    "dtr__max_features": [None,"sqrt","log2","int", "float"],
    "dtr__criterion": ["poisson"]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)


Fitting 5 folds for each of 320 candidates, totalling 1600 fits


C:\Users\DELL\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
640 fits failed out of a total of 1600.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
236 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\DELL\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\DELL\anaconda3\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 473, in fit
    self._final_estimator.fit

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('dtr',
                                        DecisionTreeRegressor(random_state=0))]),
             n_jobs=-1,
             param_grid={'dtr__criterion': ['poisson'],
                         'dtr__max_depth': [None, 2, 3, 4, 5, 6, 7, 8],
                         'dtr__max_features': [None, 'sqrt', 'log2', 'int',
                                               'float'],
                         'dtr__min_samples_leaf': [1, 2],
                         'dtr__min_samples_split': [2, 5, 10, 20]},
             scoring='r2', verbose=2)

In [41]:
print("Best Parameters:", grid.best_params_)
print("Best R2 Score:", grid.best_score_)

best_model = grid.best_estimator_

Best Parameters: {'dtr__criterion': 'poisson', 'dtr__max_depth': 4, 'dtr__max_features': None, 'dtr__min_samples_leaf': 2, 'dtr__min_samples_split': 2}
Best R2 Score: 0.8219451273816732


In [43]:
from sklearn.metrics import r2_score

y_pred = best_model.predict(X_test)
print("Test R2:", r2_score(y_test, y_pred))

Test R2: 0.8860548107843377
